# 02 — Kaggle Data Preparation (LOCAL)
## WavSent-MTL · Tasks 1.14–1.18

**Pipeline steps covered (Steps 1–3 for Kaggle):**
1. Load Nifty50 OHLCV (same file, full 2017–2024 range)
2. Load pre-computed FinBERT sentiment from kaggle1_polarity.csv + kaggle2_polarity.csv
3. Merge on trading date; fill all NaN with 0 (covers pre-2020 and gap May–Dec 2021)
4. Verify gap period May–Dec 2021 is zero-filled
5. Save → `data/processed/kaggle/merged_data.csv`

**Kaggle sentiment coverage:**
- kaggle1: 2020-01-01 to 2021-12-31
- kaggle2: 2022-01-03 to 2024-05-31
- Gap (per DECISIONS.md): May–Dec 2021 — filled with 0
- Pre-2020 (2017–2019): no coverage — filled with 0

In [1]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from config.config import CONFIG

print('CONFIG loaded.')
print('Kaggle price period:', CONFIG['price_start'], '->', CONFIG['price_end'])
print('Gap period:         ', CONFIG['gap_start'], '->', CONFIG['gap_end'])

CONFIG loaded.
Kaggle price period: 2017-01-01 -> 2024-05-31
Gap period:          2021-05-01 -> 2021-12-31


## Step 1 — Load Nifty50 OHLCV

In [2]:
from src.data.loader import load_price_data

price_df = load_price_data()

print(f'Price shape:  {price_df.shape}')
print(f'Date range:   {price_df["Date"].min().date()} -> {price_df["Date"].max().date()}')
print(f'Columns:      {price_df.columns.tolist()}')
price_df.head(3)

Price shape:  (1826, 6)
Date range:   2017-01-02 -> 2024-05-31
Columns:      ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


,Date,Open,High,Low,Close,Volume
0,2017-01-02,8210.099609,8212.000000,8133.799805,8179.50,118300.0
1,2017-01-03,8196.049805,8219.099609,8148.600098,8192.25,127300.0
2,2017-01-04,8202.650391,8218.500000,8180.899902,8190.50,132400.0


## Step 2 — Load Kaggle Sentiment (DS1 + DS2)

In [3]:
from src.data.loader import load_kaggle_sentiment

kaggle_sent = load_kaggle_sentiment()

print(f'Combined shape: {kaggle_sent.shape}')
print(f'Columns:        {kaggle_sent.columns.tolist()}')
print(f'Date range:     {kaggle_sent["date"].min()} -> {kaggle_sent["date"].max()}')
print(f'NaN count:      {kaggle_sent.isnull().sum().sum()}')
kaggle_sent.head(3)

Combined shape: (1092, 3)
Columns:        ['date', 'polarity_mean', 'polarity_max']
Date range:     2020-01-01 00:00:00 -> 2024-05-31 00:00:00
NaN count:      0


,date,polarity_mean,polarity_max
0,2020-01-01,0.165681,-0.963035
1,2020-01-02,0.201445,-0.956836
2,2020-01-03,0.079484,-0.962350


## Step 3 — Merge Price + Kaggle Sentiment

In [4]:
from src.data.loader import merge_kaggle

df_kaggle = merge_kaggle(price_df, kaggle_sent)

print(f'Merged shape: {df_kaggle.shape}')
print(f'Columns:      {df_kaggle.columns.tolist()}')
print(f'Date range:   {df_kaggle["Date"].min().date()} -> {df_kaggle["Date"].max().date()}')
df_kaggle.head(3)

Merged shape: (1826, 8)
Columns:      ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'polarity_mean', 'polarity_max']
Date range:   2017-01-02 -> 2024-05-31


,Date,Open,High,Low,Close,Volume,polarity_mean,polarity_max
0,2017-01-02,8210.099609,8212.000000,8133.799805,8179.50,118300.0,0.0,0.0
1,2017-01-03,8196.049805,8219.099609,8148.600098,8192.25,127300.0,0.0,0.0
2,2017-01-04,8202.650391,8218.500000,8180.899902,8190.50,132400.0,0.0,0.0


## Task 1.16 — Verify Gap May–Dec 2021 Filled with 0

In [5]:
# Check 1: No NaN in either sentiment column
nan_mean = df_kaggle['polarity_mean'].isnull().sum()
nan_max  = df_kaggle['polarity_max'].isnull().sum()
assert nan_mean == 0, f'polarity_mean has {nan_mean} NaN'
assert nan_max  == 0, f'polarity_max has {nan_max} NaN'
print(f'NaN polarity_mean: {nan_mean}  OK')
print(f'NaN polarity_max:  {nan_max}  OK')

# Check 2: Gap period is zero-filled
gap_start = pd.to_datetime(CONFIG['gap_start'])
gap_end   = pd.to_datetime(CONFIG['gap_end'])
gap = df_kaggle[(df_kaggle['Date'] >= gap_start) & (df_kaggle['Date'] <= gap_end)]
print(f'Gap rows: {len(gap)}')
assert (gap['polarity_mean'] == 0).all(), 'Gap polarity_mean not zero-filled'
assert (gap['polarity_max']  == 0).all(), 'Gap polarity_max not zero-filled'
print(f'Gap zero-fill verified: polarity_mean={gap["polarity_mean"].unique()}  OK')
print(f'Gap zero-fill verified: polarity_max={gap["polarity_max"].unique()}    OK')

# Check 3: Pre-2020 also zero-filled (no kaggle sentiment before 2020)
pre2020 = df_kaggle[df_kaggle['Date'] < '2020-01-01']
print(f'Pre-2020 rows: {len(pre2020)}')
if len(pre2020) > 0:
    assert (pre2020['polarity_mean'] == 0).all(), 'Pre-2020 polarity_mean not zero-filled'
    assert (pre2020['polarity_max']  == 0).all(), 'Pre-2020 polarity_max not zero-filled'
    print('Pre-2020 zero-fill: OK')

NaN polarity_mean: 0  OK
NaN polarity_max:  0  OK
Gap rows: 168
Gap zero-fill verified: polarity_mean=[0.]  OK
Gap zero-fill verified: polarity_max=[0.]    OK
Pre-2020 rows: 734
Pre-2020 zero-fill: OK


## Task 1.17 — Save merged_data.csv

In [6]:
out_dir = os.path.join(project_root, CONFIG['kaggle_processed_dir'])
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, 'merged_data.csv')
df_kaggle.to_csv(out_path, index=False)

# Reload and verify
df_check = pd.read_csv(out_path)
assert df_check.shape == df_kaggle.shape, 'Save/reload shape mismatch'

print(f'Saved:  {out_path}')
print(f'Shape:  {df_kaggle.shape}')
print('Reload: OK')
df_check.tail(3)

Saved:  d:/WavSent-MTL/data/processed/kaggle/merged_data.csv
Shape:  (1826, 8)
Reload: OK


,Date,Open,High,Low,Close,Volume,polarity_mean,polarity_max
1823,2024-05-29,22762.75000,22825.50,22685.44922,22704.69922,269900.0,0.004811,-0.967315
1824,2024-05-30,22617.44922,22705.75,22417.00000,22488.65039,373400.0,0.000000,0.000000
1825,2024-05-31,22568.09961,22653.75,22465.09961,22530.69922,572100.0,0.000000,0.000000


## Summary

In [7]:
print('=' * 50)
print('Notebook 02 -- Kaggle Data Prep: COMPLETE')
print('=' * 50)
print(f'  Rows:            {len(df_kaggle)}')
print(f'  Date range:      {df_kaggle["Date"].min().date()} -> {df_kaggle["Date"].max().date()}')
print(f'  polarity_mean:   min={df_kaggle["polarity_mean"].min():.4f}  max={df_kaggle["polarity_mean"].max():.4f}')
print(f'  polarity_max:    min={df_kaggle["polarity_max"].min():.4f}  max={df_kaggle["polarity_max"].max():.4f}')
print(f'  Zero-filled:     {(df_kaggle["polarity_mean"] == 0.0).sum()} rows (pre-2020 + gap)')
print(f'  Output:          {out_path}')
print()
print('Next: run 03_feature_engineering.ipynb')

Notebook 02 -- Kaggle Data Prep: COMPLETE
  Rows:            1826
  Date range:      2017-01-02 -> 2024-05-31
  polarity_mean:   min=-0.2504  max=0.2821
  polarity_max:    min=-0.9690  max=0.9351
  Zero-filled:     953 rows (pre-2020 + gap)
  Output:          d:/WavSent-MTL/data/processed/kaggle/merged_data.csv

Next: run 03_feature_engineering.ipynb
